In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier


# Detect the repository root
current_dir = Path.cwd()

if (current_dir / "data").exists():
    repository_root = current_dir
elif (current_dir.parent / "data").exists():
    repository_root = current_dir.parent
else:
    raise FileNotFoundError(
        "The repository data directory could not be found. "
        "Run the notebook from the repository root or from the notebooks directory."
    )


# Dataset path
data_path = repository_root / "data" / "Dataset_Consumo_Label_v1.xlsx"

if not data_path.exists():
    raise FileNotFoundError(
        "Dataset not found. Place 'Dataset_Consumo_Label_v1.xlsx' "
        "inside the repository data directory."
    )


# Load the dataset
df = pd.read_excel(data_path)


# Define the input features and target
feature_columns = [
    "Consumo",
    "Morning Shift",
    "Afternoon Shift",
    "Night Shift"
]

target_column = "Label"

required_columns = feature_columns + [target_column]

missing_columns = [
    column for column in required_columns
    if column not in df.columns
]

if missing_columns:
    raise KeyError(
        "The dataset is missing the following required columns: "
        f"{missing_columns}"
    )


# Remove observations with missing values
df = df.dropna(subset=required_columns).copy()

if df.empty:
    raise ValueError(
        "No valid observations remain after removing missing values."
    )


# Prepare features and class labels
X = df[feature_columns].copy()

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(df[target_column])


# Configure stratified 10-fold cross-validation
cv = StratifiedKFold(
    n_splits=10,
    shuffle=True,
    random_state=42
)

scoring = {
    "accuracy": "accuracy",
    "macro_f1": "f1_macro"
}


# Define the supervised-learning models
models = {
    "Decision Tree": DecisionTreeClassifier(
        random_state=42
    ),
    "Random Forest": RandomForestClassifier(
        random_state=42
    ),
    "XGBoost": XGBClassifier(
        random_state=42,
        eval_metric="mlogloss"
    ),
    "k-Nearest Neighbors": KNeighborsClassifier(),
    "Support Vector Machine (SVM)": SVC(
        random_state=42
    ),
    "Logistic Regression": LogisticRegression(
        max_iter=1000,
        random_state=42
    )
}


# Evaluate the models
results = []

for model_name, model in models.items():

    pipeline = Pipeline([
        ("scaler", MinMaxScaler()),
        ("classifier", model)
    ])

    cv_results = cross_validate(
        pipeline,
        X,
        y,
        cv=cv,
        scoring=scoring,
        return_train_score=False
    )

    results.append({
        "Model": model_name,
        "Accuracy Mean": cv_results["test_accuracy"].mean(),
        "Accuracy Std": cv_results["test_accuracy"].std(),
        "Macro F1 Mean": cv_results["test_macro_f1"].mean(),
        "Macro F1 Std": cv_results["test_macro_f1"].std()
    })


# Create the formatted table
results_df = pd.DataFrame(results)

table_df = pd.DataFrame({
    "Model": results_df["Model"],
    "Accuracy": results_df.apply(
        lambda row:
        f"{row['Accuracy Mean']:.3f} ± {row['Accuracy Std']:.3f}",
        axis=1
    ),
    "Macro F1-Score": results_df.apply(
        lambda row:
        f"{row['Macro F1 Mean']:.3f} ± {row['Macro F1 Std']:.3f}",
        axis=1
    )
})


# Display Table VI
table_df

/opt/anaconda3/lib/python3.7/site-packages/xgboost/sklearn.py:1224: UserWarning: The use of label encoder in XGBClassifier is deprecated and will be removed in a future release. To remove this warning, do the following: 1) Pass option use_label_encoder=False when constructing XGBClassifier object; and 2) Encode your labels (y) as integers starting with 0, i.e. 0, 1, 2, ..., [num_class - 1].
  warnings.warn(label_encoder_deprecation_msg, UserWarning)
/opt/anaconda3/lib/python3.7/site-packages/xgboost/sklearn.py:1224: UserWarning: The use of label encoder in XGBClassifier is deprecated and will be removed in a future release. To remove this warning, do the following: 1) Pass option use_label_encoder=False when constructing XGBClassifier object; and 2) Encode your labels (y) as integers starting with 0, i.e. 0, 1, 2, ..., [num_class - 1].
  warnings.warn(label_encoder_deprecation_msg, UserWarning)
/opt/anaconda3/lib/python3.7/site-packages/xgboost/sklearn.py:1224: UserWarning: The use of 

,Model,Accuracy,Macro F1-Score
0,Decision Tree,0.988 ± 0.025,0.969 ± 0.079
1,Random Forest,0.988 ± 0.025,0.969 ± 0.079
2,XGBoost,0.988 ± 0.025,0.969 ± 0.079
3,k-Nearest Neighbors,0.931 ± 0.065,0.872 ± 0.136
4,Support Vector Machine (SVM),0.906 ± 0.075,0.871 ± 0.115
5,Logistic Regression,0.698 ± 0.096,0.499 ± 0.114
